# 02 — DataFrame Deep Dive

DataFrame is a 2D labeled data structure with columns of potentially different dtypes.
This notebook covers construction patterns, attributes, column operations, method chaining,
and the internals every data engineer needs to know.

In [ ]:
import pandas as pd
import numpy as np
from io import StringIO

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

## 1. DataFrame Construction Patterns

In [ ]:
# 1a. From dict of lists — most common
df_dict = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol'],
    'salary': [90000, 85000, 110000],
    'dept': ['Engineering', 'Sales', 'Engineering']
})
print(df_dict)

In [ ]:
# 1b. From list of dicts — each dict = one row
records = [
    {'name': 'Alice', 'salary': 90000, 'dept': 'Engineering'},
    {'name': 'Bob',   'salary': 85000, 'dept': 'Sales'},
    {'name': 'Carol', 'salary': 110000}  # missing dept → NaN
]
df_records = pd.DataFrame(records)
print(df_records)

In [ ]:
# 1c. From numpy array
arr = np.random.randint(50000, 150000, size=(4, 3))
df_np = pd.DataFrame(arr, columns=['Q1', 'Q2', 'Q3'],
                     index=['Alice', 'Bob', 'Carol', 'Dave'])
print(df_np)

In [ ]:
# 1d. From CSV string (useful for quick tests and demos)
csv_data = """name,salary,dept
Alice,90000,Engineering
Bob,85000,Sales
Carol,110000,Engineering
"""
df_csv = pd.read_csv(StringIO(csv_data))
print(df_csv)

## 2. The Primary Dataset — Employees

In [ ]:
np.random.seed(42)
n = 200

employees = pd.DataFrame({
    'emp_id':           range(1001, 1001 + n),
    'name':             [f'Employee_{i:03d}' for i in range(n)],
    'department':       np.random.choice(['Engineering', 'Sales', 'HR', 'Marketing', 'Finance'], n),
    'salary':           np.random.randint(40000, 150000, n),
    'experience_years': np.random.randint(1, 20, n),
    'join_date':        pd.date_range('2010-01-01', periods=n, freq='W'),
    'rating':           np.random.choice([1.0, 2.0, 3.0, 4.0, 5.0], n),
    'city':             np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Chennai'], n),
    'active':           np.random.choice([True, False], n, p=[0.85, 0.15])
})

employees.head()

## 3. Core Attributes

In [ ]:
print(f"shape:   {employees.shape}")      # (rows, cols)
print(f"ndim:    {employees.ndim}")       # always 2
print(f"size:    {employees.size}")       # total elements
print(f"columns: {employees.columns.tolist()}")
print(f"index:   {employees.index}")

In [ ]:
# dtypes per column
print(employees.dtypes)

In [ ]:
# info() — most useful for quick inspection: dtypes + non-null counts + memory
employees.info()

In [ ]:
# describe() — default: numeric columns only
print(employees.describe().round(1))

In [ ]:
# describe(include='all') — includes object/category columns
print(employees[['department', 'city', 'active']].describe())

## 4. Column Selection

In [ ]:
# Single column — returns Series
salaries = employees['salary']
print(type(salaries), salaries.shape)

# Multiple columns — returns DataFrame
subset = employees[['name', 'department', 'salary']]
print(type(subset), subset.shape)

In [ ]:
# Dot notation — AVOID in production code
# Works only if column name is a valid Python identifier AND not a method name
# BREAKS silently when column name has spaces or shadows a method
print(employees.salary.mean())    # works
# employees.join_date              # works, but 'join' is a method — dangerous

## 5. Adding Columns

In [ ]:
# Direct assignment — appends to end
employees['annual_bonus'] = (employees['salary'] * 0.10).round(0)
employees['total_comp']   = employees['salary'] + employees['annual_bonus']

employees[['name', 'salary', 'annual_bonus', 'total_comp']].head()

In [ ]:
# insert() — precise column position control
employees.insert(
    loc=4,           # column position
    column='salary_k',
    value=(employees['salary'] / 1000).round(1)
)
employees.columns.tolist()

## 6. Renaming Columns

In [ ]:
# rename() with dict — rename specific columns
renamed = employees.rename(columns={
    'experience_years': 'exp_yrs',
    'annual_bonus':     'bonus'
})
print(renamed.columns.tolist())

In [ ]:
# rename() with function — transform all column names
all_caps = employees.rename(columns=str.upper)
print(all_caps.columns.tolist())

# Strip whitespace from all column names — common after reading messy CSVs
employees.columns = employees.columns.str.strip().str.lower().str.replace(' ', '_')
print(employees.columns.tolist())

## 7. Dropping Rows and Columns

In [ ]:
# Drop by column name — axis=1 or axis='columns'
temp = employees.drop(columns=['salary_k'])
print("Columns after drop:", temp.columns.tolist())

# Drop by row index — axis=0 (default)
temp2 = employees.drop(index=[0, 1, 2])
print(f"Rows after drop: {len(temp2)}")

In [ ]:
# inplace=True vs returning new df
# Best practice: avoid inplace=True — return new df for method chaining
employees.drop(columns=['salary_k'], inplace=True)
print(employees.columns.tolist())

## 8. head(), tail(), sample()

In [ ]:
print(employees.head(3))
print()
print(employees.tail(3))
print()
# sample() — reproducible with random_state
print(employees.sample(3, random_state=42))

## 9. Transpose

In [ ]:
# .T transposes — rows become columns, columns become rows
# Useful for viewing wide DataFrames in narrow screens
print(employees.head(3).T)

## 10. Index Operations

In [ ]:
# set_index() — use a column as row labels
emp_indexed = employees.set_index('emp_id')
print(emp_indexed.head(3))

In [ ]:
# reset_index() — move index back to column
back = emp_indexed.reset_index()
print(back.head(3))

# drop=True — discard index instead of returning it to column
emp_fresh = emp_indexed.reset_index(drop=True)
print(emp_fresh.head(3))

In [ ]:
# reindex() — reorder rows or fill missing labels with NaN
small = employees.set_index('name').head(5)
new_order = ['Employee_004', 'Employee_002', 'Employee_000', 'Employee_099']
reindexed = small.reindex(new_order)  # Employee_099 doesn't exist → NaN row
print(reindexed[['department', 'salary']])

## 11. Converting DataFrame

In [ ]:
small_df = employees[['name', 'department', 'salary']].head(3)

# to_dict() — multiple orientations
print("records:", small_df.to_dict('records'))  # list of dicts — most common
print()
print("index:",   small_df.to_dict('index'))    # {idx: {col: val}}

In [ ]:
# to_numpy() vs .values
# to_numpy() is preferred — .values may not honor dtypes for nullable integer arrays
arr1 = employees[['salary', 'experience_years']].head(3).to_numpy()
arr2 = employees[['salary', 'experience_years']].head(3).values
print(arr1)
print(f"type: {type(arr1)}, dtype: {arr1.dtype}")

## 12. Iterating — and Why to Avoid It

In [ ]:
import time

test_df = employees[['salary', 'experience_years']].copy()

# Method 1: iterrows() — slowest, creates Series per row
start = time.time()
totals_iterrows = []
for _, row in test_df.iterrows():
    totals_iterrows.append(row['salary'] + row['experience_years'] * 1000)
t_iterrows = time.time() - start

# Method 2: itertuples() — faster, named tuples
start = time.time()
totals_itertuples = []
for row in test_df.itertuples():
    totals_itertuples.append(row.salary + row.experience_years * 1000)
t_itertuples = time.time() - start

# Method 3: Vectorized — fastest, no loop
start = time.time()
totals_vectorized = (test_df['salary'] + test_df['experience_years'] * 1000).tolist()
t_vectorized = time.time() - start

print(f"iterrows:    {t_iterrows*1000:.2f}ms")
print(f"itertuples:  {t_itertuples*1000:.2f}ms")
print(f"vectorized:  {t_vectorized*1000:.2f}ms")
print(f"\nVectorized is ~{t_iterrows/t_vectorized:.0f}x faster than iterrows")

## 13. assign() — Functional Column Addition

In [ ]:
# assign() returns a new DataFrame — perfect for method chains
# Later columns can reference earlier ones in the same call (Pandas 0.23+)
result = (
    employees
    .assign(
        salary_band=lambda df: pd.cut(
            df['salary'],
            bins=[0, 60000, 100000, 200000],
            labels=['Junior', 'Mid', 'Senior']
        ),
        comp_per_year=lambda df: df['total_comp'] / df['experience_years']
    )
    [['name', 'salary', 'salary_band', 'total_comp', 'comp_per_year']]
    .head(6)
)
print(result)

## 14. pipe() — Clean Method Chaining

In [ ]:
def add_comp_ratio(df, base_col='salary', exp_col='experience_years'):
    """Compute salary per year of experience."""
    df = df.copy()
    df['comp_ratio'] = df[base_col] / df[exp_col]
    return df

def flag_senior(df, years_threshold=10):
    """Mark employees as senior based on experience."""
    df = df.copy()
    df['is_senior'] = df['experience_years'] >= years_threshold
    return df

# pipe() keeps the chain readable
processed = (
    employees
    .pipe(add_comp_ratio)
    .pipe(flag_senior, years_threshold=10)
    [['name', 'salary', 'experience_years', 'comp_ratio', 'is_senior']]
    .head(8)
)
print(processed.round(1))

## 15. DataFrame Equality and Copying

In [ ]:
df_a = employees[['name', 'salary']].head(3)
df_b = employees[['name', 'salary']].head(3)

# == is element-wise, equals() is whole-DataFrame
print("equals():", df_a.equals(df_b))
print("== gives DataFrame of bools:")
print(df_a == df_b)

In [ ]:
# Deep copy vs shallow copy
deep   = df_a.copy(deep=True)   # independent copy — default
shallow = df_a.copy(deep=False)  # shares data arrays — modifying one may affect other

print(f"deep copy shares memory:    {np.shares_memory(df_a['salary'].values, deep['salary'].values)}")
print(f"shallow copy shares memory: {np.shares_memory(df_a['salary'].values, shallow['salary'].values)}")

## 16. empty, any(), all()

In [ ]:
empty_df = pd.DataFrame()
print(f"empty_df.empty: {empty_df.empty}")
print(f"employees.empty: {employees.empty}")

# any() and all() on boolean masks across columns
bool_df = employees[['salary', 'experience_years']] > 80000
print("\nany() per column (any row > 80k):")
print(bool_df.any())
print("\nall() per row (all cols > 80k):")
print(bool_df.all(axis=1).head())

## 17. Column Reordering

In [ ]:
# Method 1: explicit column list
priority_cols = ['emp_id', 'name', 'department', 'salary', 'total_comp']
remaining = [c for c in employees.columns if c not in priority_cols]
reordered = employees[priority_cols + remaining]
print(reordered.columns.tolist())

In [ ]:
# Method 2: move a column to front with insert + drop
def move_column_front(df, col):
    return df.insert(0, col, df.pop(col)) or df

# Cleaner version — return new df
def reorder_cols(df, first_cols):
    rest = [c for c in df.columns if c not in first_cols]
    return df[first_cols + rest]

print(reorder_cols(employees, ['name', 'department']).columns.tolist())

## 18. select_dtypes()

In [ ]:
# Numeric columns only
numeric_df = employees.select_dtypes(include='number')
print("Numeric columns:", numeric_df.columns.tolist())

# Object (string) columns
str_df = employees.select_dtypes(include='object')
print("Object columns: ", str_df.columns.tolist())

# Exclude datetime
no_dt = employees.select_dtypes(exclude=['datetime64'])
print("Excl datetime:  ", no_dt.columns.tolist())

## 19. Memory Usage

In [ ]:
mem_usage = employees.memory_usage(deep=True)  # deep=True for object columns
print("Memory per column (bytes):")
print(mem_usage)
print(f"\nTotal: {mem_usage.sum() / 1024:.1f} KB")

## 20. items() and to_records()

In [ ]:
# items() — iterates (column_name, Series)
for col_name, col_series in employees[['salary', 'experience_years']].items():
    print(f"{col_name}: mean={col_series.mean():.0f}, std={col_series.std():.0f}")

In [ ]:
# to_records() — structured numpy array; useful for interfacing with C libraries
records_arr = employees[['name', 'salary']].head(3).to_records(index=False)
print(records_arr)
print(f"dtype: {records_arr.dtype}")

## 21. Real-World Pattern — Full Cleaning Pipeline

In [ ]:
# Simulate a raw messy dataset
raw = pd.DataFrame({
    ' Name ':     ['Alice ', ' Bob', 'Carol'],
    'Salary':     ['90000', '85000', 'N/A'],
    'Department': ['Engineering', 'SALES', None],
    'StartDate':  ['01-Jan-2020', '15-Mar-2019', '2021/06/01']
})

# Full pipeline with method chaining
cleaned = (
    raw
    .rename(columns=lambda c: c.strip().lower().replace(' ', '_'))
    .assign(
        name        = lambda df: df['name'].str.strip(),
        salary      = lambda df: pd.to_numeric(df['salary'], errors='coerce'),
        department  = lambda df: df['department'].str.title().fillna('Unknown'),
        start_date  = lambda df: pd.to_datetime(df['startdate'], infer_datetime_format=True)
    )
    .drop(columns=['startdate'])
)

print(cleaned)
print()
print(cleaned.dtypes)